# Part 3 - Multi-Agent Supervisor

A LangGraph supervisor routes each query to a Revenue Agent, an Expenditure Agent, or both. A synthesis node combines source-backed findings. The trace records every routing and agent decision.

In [1]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/input/fy2024_analysis_of_revenue_and_expenditure.pdf").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_PATH = PROJECT_ROOT / "data/input/fy2024_analysis_of_revenue_and_expenditure.pdf"
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Assessment PDF not found: {PDF_PATH}")

SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

import htx_fde_ai_assessment.part3_supervisor as part3_supervisor
import htx_fde_ai_assessment.gemini_client as gemini_client

importlib.reload(part3_supervisor)
importlib.reload(gemini_client)
load_part3_source_context = part3_supervisor.load_part3_source_context
build_part3_graph = part3_supervisor.build_part3_graph
run_part3_query = part3_supervisor.run_part3_query
load_gemini_config = gemini_client.load_gemini_config
create_gemini_chat_model = gemini_client.create_gemini_chat_model

GEMINI_CONFIG = load_gemini_config()
if not GEMINI_CONFIG["api_key_configured"]:
    raise RuntimeError("Set GEMINI_API_KEY in the repository-root .env file.")

PART3_SOURCE_CONTEXT = load_part3_source_context(PDF_PATH)
PART3_LLM = create_gemini_chat_model(temperature=0)
PART3_GRAPH = build_part3_graph(PART3_LLM)
print(f"Gemini model ready: {GEMINI_CONFIG['model']}")
print("Loaded PDF pages:", sorted(PART3_SOURCE_CONTEXT))

Gemini model ready: gemini-3.8-flash
Loaded PDF pages: [16, 18, 20, 37]


## Demo Queries

Revenue evidence comes from pages 16 and 37. Fund evidence comes from pages 18 and 20. Page 16 contains FY2024 estimates. The source states a government injection for the Future Energy Fund but does not identify a separate tax or recurring revenue stream; the agents must not infer one.

Choose one query below. Each run makes multiple Gemini requests; the combined query routes to both agents.

In [2]:
PART3_DEMO_QUERIES = {
    "revenue_only": "What are the key government revenue streams?",
    "fund_only": "How will the Budget for the Future Energy Fund be supported?",
    "combined": (
        "What are the key government revenue streams, and how will the "
        "Budget for the Future Energy Fund be supported?"
    ),
}

PART3_QUERY_TO_RUN = "combined"
if PART3_QUERY_TO_RUN not in PART3_DEMO_QUERIES:
    raise ValueError(f"Unknown Part 3 demo query: {PART3_QUERY_TO_RUN}")

PART3_RESULT = run_part3_query(
    PART3_GRAPH,
    PART3_DEMO_QUERIES[PART3_QUERY_TO_RUN],
    PART3_SOURCE_CONTEXT,
)

display({
    "query": PART3_RESULT["query"],
    "answer": PART3_RESULT["answer"],
    "selected_agents": PART3_RESULT["selected_agents"],
    "citations": PART3_RESULT["citations"],
    "assumptions": PART3_RESULT["assumptions"],
})
display(PART3_RESULT["trace"])

{'query': 'What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?',
 'answer': 'The main components of government Operating Revenue are Corporate Income Tax, Personal Income Tax, and Goods and Services Tax (GST) (page 37). For FY2024, total operating revenue is estimated at $108.64 billion, with key streams including Corporate Income Tax (estimated at $28.03 billion) and Goods and Services Tax (estimated at $19.39 billion) (page 16).\n\nThe Budget supports the Future Energy Fund with an initial injection of $5.0 billion to invest in critical infrastructure for the energy transition (page 18). In FY2024, this funding is provided via Top-ups to Endowment and Trust Funds, with an estimated top-up of $5.00 billion ($5,000 million) (page 16, page 20). The provided sources do not specify any dedicated revenue stream for the fund.',
 'selected_agents': ['revenue', 'expenditure'],
 'citations': [{'source_page': 37,
   'quote': 'Government 

[{'node': 'supervisor',
  'selected_agents': ['revenue', 'expenditure'],
  'reason': 'The query asks about key government revenue streams, which falls under revenue, and how a specific fund budget (the Future Energy Fund) is financed and supported, requiring input on both funding mechanisms and expenditure allocations.'},
 {'node': 'revenue_agent', 'status': 'completed', 'evidence_pages': [16, 37]},
 {'node': 'expenditure_agent',
  'status': 'completed',
  'evidence_pages': [18, 20]},
 {'node': 'synthesis',
  'status': 'completed',
  'agents_used': ['revenue', 'expenditure'],
  'citation_pages': [16, 18, 20, 37]}]